In [ ]:
from alpaca.data.historical import StockHistoricalDataClient as shdc
from alpaca.data.requests import StockBarsRequest
from alpaca.data.timeframe import TimeFrame
from datetime import datetime
import os

import getpass
import psycopg

stock_c = shdc(os.environ['API_KEY'],os.environ['SECRET_KEY'])

# Obtain a database connection
db_host = 'localhost'
db_user = 'postgres'
db_name = 'securities_master'
db_pass = getpass.getpass('db password')

In [ ]:
def get_vendor_id(vendor_name='ALPACA'):
    con = psycopg.connect(f"host={db_host} user={db_user} password={db_pass} dbname={db_name}") 
    with con:
        cur = con.cursor()
        cur.execute(f"SELECT id FROM data_vendor WHERE name='{vendor_name}'")
        idx = cur.fetchall()
    return idx[0][0]
    
def insert_data_into_db(data_vendor_id, symbol, data):
    """
    Takes a list of tuples of daily data and adds it to the
    MySQL database. Appends the vendor ID and symbol ID to the data.
    daily_data: List of tuples of the OHLC data (with
    adj_close and volume)
    """
    # Create the time now
    now = datetime.now()
    
    # Amend the data to include the vendor ID and symbol ID
    daily_data = [(data_vendor_id, t,
                   d[0],d[1].open, d[1].high, d[1].low, d[1].close, d[1].volume, d[1].trade_count, d[1].vwap) for d in data.iterrows()]
    
    # Create the insert strings
    column_str = """data_vendor_id, symbol,
                    price_date, open, high, low, close, volume, trade_count, vwap"""
    
    insert_str = ("%s, " * 10)[:-2]
    
    final_str = "INSERT INTO price_data (%s) VALUES (%s)" % (column_str, insert_str)
    # Using the MySQL connection, carry out an INSERT INTO for every symbol
    con = psycopg.connect(f"host={db_host} user={db_user} password={db_pass} dbname={db_name}") 
    with con:
        cur = con.cursor()
        cur.executemany(final_str, daily_data)

In [ ]:
tickers = ['SPY','QQQ']
request_params = StockBarsRequest(
                        symbol_or_symbols= tickers,
                        timeframe=TimeFrame.Minute,
                        start=datetime(2022, 2, 1),
                        end=datetime(2025, 3, 30)
                 )

bars = stock_c.get_stock_bars(request_params).df
bars

In [ ]:
for ticker in tickers:
    print(ticker, bars.loc[ticker].index.min(), bars.loc[ticker].index.max(), len(bars.loc[ticker].dropna()))

In [ ]:
# Loop over the tickers and insert the daily historical data into the database
#tickers = bars.index.unique(level=0)
vendor_id = get_vendor_id('ALPACA')  

for i, t in enumerate(tickers):
    print("Adding data for %s: %s out of %s" %(t, i+1, len(tickers)))
    insert_data_into_db(vendor_id, t, bars.loc[(t)])
    
print("Successfully added data to DB.")